In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# DEBUG ESTRUCTURA DE DATOS - ANÁLISIS COMPLETO
# =====================================================================

class DataStructureDebugger:
    """Debugger completo para entender la estructura de datos real."""
    
    def __init__(self, match_path):
        """
        Args:
            match_path: Path al directorio del partido específico
        """
        self.match_path = Path(match_path)
        self.csv_path = self.match_path / 'csv'
        self.data_loaded = {}
        
    def load_all_csvs(self):
        """Carga todos los CSVs disponibles para análisis."""
        csv_files = {
            'events': 'events.csv',
            'players': 'players.csv', 
            'formations': 'formations_timeline.csv',
            'positions': 'player_positions_timeline.csv',
            'match_meta': 'match_meta.csv'
        }
        
        print("CARGANDO DATOS DEL PARTIDO")
        print("=" * 50)
        
        for key, filename in csv_files.items():
            file_path = self.csv_path / filename
            if file_path.exists():
                try:
                    df = pd.read_csv(file_path)
                    self.data_loaded[key] = df
                    print(f"✓ {key}: {len(df)} filas, {len(df.columns)} columnas")
                except Exception as e:
                    print(f"✗ Error cargando {key}: {e}")
            else:
                print(f"✗ No encontrado: {filename}")
        
        return self.data_loaded
    
    def analyze_events_structure(self):
        """Analiza estructura del CSV de eventos."""
        if 'events' not in self.data_loaded:
            print("No hay datos de eventos cargados")
            return
            
        events = self.data_loaded['events']
        
        print(f"\nANÁLISIS DE EVENTOS")
        print("=" * 30)
        
        # Información básica
        print(f"Total eventos: {len(events)}")
        print(f"Columnas: {list(events.columns)}")
        
        # Verificar datos de tiempo
        if 'minute' in events.columns:
            print(f"Minutos: {events['minute'].min()}-{events['minute'].max()}")
            
        # Verificar coordenadas
        coord_cols = ['x', 'y', 'endX', 'endY']
        available_coords = [col for col in coord_cols if col in events.columns]
        print(f"Coordenadas disponibles: {available_coords}")
        
        if 'x' in events.columns and 'y' in events.columns:
            print(f"Rango X: {events['x'].min():.1f} - {events['x'].max():.1f}")
            print(f"Rango Y: {events['y'].min():.1f} - {events['y'].max():.1f}")
        
        # Tipos de eventos más frecuentes
        if 'typeName' in events.columns:
            print(f"\nTipos de eventos más frecuentes:")
            top_events = events['typeName'].value_counts().head(10)
            for event_type, count in top_events.items():
                print(f"  {event_type}: {count}")
        
        # Resultados de eventos
        if 'outcomeName' in events.columns:
            print(f"\nResultados de eventos:")
            outcomes = events['outcomeName'].value_counts()
            for outcome, count in outcomes.items():
                print(f"  {outcome}: {count}")
                
        return events
    
    def analyze_players_structure(self):
        """Analiza estructura del CSV de jugadores."""
        if 'players' not in self.data_loaded:
            print("No hay datos de jugadores cargados")
            return
            
        players = self.data_loaded['players']
        
        print(f"\nANÁLISIS DE JUGADORES")
        print("=" * 30)
        
        print(f"Total jugadores: {len(players)}")
        print(f"Columnas: {list(players.columns)}")
        
        # Verificar posiciones
        if 'position' in players.columns:
            positions = players['position'].value_counts()
            print(f"\nPosiciones:")
            for pos, count in positions.items():
                print(f"  {pos}: {count}")
                
        # Verificar equipos
        if 'teamId' in players.columns:
            teams = players['teamId'].value_counts()
            print(f"\nEquipos (por ID):")
            for team_id, count in teams.items():
                print(f"  Equipo {team_id}: {count} jugadores")
        
        # Detectar titulares vs suplentes
        starters = players[players['position'] != 'sub'] if 'position' in players.columns else players
        subs = players[players['position'] == 'sub'] if 'position' in players.columns else pd.DataFrame()
        
        print(f"\nTitulares detectados: {len(starters)}")
        print(f"Suplentes detectados: {len(subs)}")
        
        return players
    
    def analyze_formations_timeline(self):
        """Analiza timeline de formaciones."""
        if 'formations' not in self.data_loaded:
            print("No hay datos de formaciones")
            return
            
        formations = self.data_loaded['formations']
        
        print(f"\nANÁLISIS DE FORMACIONES")
        print("=" * 30)
        
        print(f"Registros de formación: {len(formations)}")
        print(f"Columnas: {list(formations.columns)}")
        
        # Mostrar muestra de datos
        if len(formations) > 0:
            print(f"\nMuestra de datos:")
            print(formations.head())
            
        return formations
    
    def analyze_positions_timeline(self):
        """Analiza timeline de posiciones de jugadores."""
        if 'positions' not in self.data_loaded:
            print("No hay datos de posiciones timeline")
            return
            
        positions = self.data_loaded['positions']
        
        print(f"\nANÁLISIS DE POSICIONES TIMELINE")
        print("=" * 30)
        
        print(f"Registros de posición: {len(positions)}")
        print(f"Columnas: {list(positions.columns)}")
        
        # Mostrar muestra
        if len(positions) > 0:
            print(f"\nMuestra de datos:")
            print(positions.head())
            
        return positions
    
    def analyze_match_meta(self):
        """Analiza metadatos del partido."""
        if 'match_meta' not in self.data_loaded:
            print("No hay metadatos del partido")
            return
            
        meta = self.data_loaded['match_meta']
        
        print(f"\nMETADATOS DEL PARTIDO")
        print("=" * 30)
        
        print(f"Información disponible: {len(meta)} registros")
        print(f"Columnas: {list(meta.columns)}")
        
        # Mostrar información clave
        if len(meta) > 0:
            print(f"\nInformación del partido:")
            for col in meta.columns:
                if len(meta[col].dropna()) > 0:
                    print(f"  {col}: {meta[col].iloc[0]}")
                    
        return meta
    
    def test_player_events_merge(self):
        """Prueba el merge entre eventos y jugadores."""
        if 'events' not in self.data_loaded or 'players' not in self.data_loaded:
            print("No se pueden realizar pruebas de merge sin datos de eventos y jugadores")
            return
            
        events = self.data_loaded['events']
        players = self.data_loaded['players']
        
        print(f"\nPRUEBA DE MERGE EVENTOS-JUGADORES")
        print("=" * 40)
        
        # Verificar claves de merge
        events_player_col = 'playerId' if 'playerId' in events.columns else None
        players_id_col = 'player_id' if 'player_id' in players.columns else 'playerId'
        
        if not events_player_col:
            print("No se encontró columna playerId en eventos")
            return
            
        if players_id_col not in players.columns:
            print(f"No se encontró columna {players_id_col} en jugadores")
            return
        
        # Hacer merge
        merged = events.merge(
            players[['player_id', 'player_name', 'position']] if 'player_id' in players.columns 
            else players[['playerId', 'player_name', 'position']],
            left_on=events_player_col,
            right_on=players_id_col,
            how='left'
        )
        
        # Estadísticas del merge
        total_events = len(events)
        matched_events = len(merged.dropna(subset=['player_name']))
        
        print(f"Eventos totales: {total_events}")
        print(f"Eventos con jugador asignado: {matched_events}")
        print(f"Tasa de éxito: {(matched_events/total_events)*100:.1f}%")
        
        # Jugadores con más eventos
        if matched_events > 0:
            player_events = merged.groupby('player_name').size().sort_values(ascending=False)
            print(f"\nJugadores con más eventos:")
            for player, count in player_events.head(10).items():
                print(f"  {player}: {count}")
                
        return merged
    
    def identify_starters_by_events(self):
        """Identifica titulares basándose en número de eventos y tiempo de juego."""
        merged = self.test_player_events_merge()
        if merged is None:
            return
            
        print(f"\nIDENTIFICACIÓN DE TITULARES")
        print("=" * 30)
        
        # Contar eventos por jugador
        player_stats = merged.groupby('player_name').agg({
            'eventId': 'count',
            'minute': ['min', 'max'],
            'position': 'first'
        }).round(1)
        
        player_stats.columns = ['total_events', 'first_minute', 'last_minute', 'position']
        player_stats['playing_time'] = player_stats['last_minute'] - player_stats['first_minute']
        
        # Filtrar suplentes obvios
        starters_candidates = player_stats[
            (player_stats['total_events'] >= 10) &  # Al menos 10 eventos
            (player_stats['position'] != 'sub')     # No marcados como suplente
        ].sort_values('total_events', ascending=False)
        
        print(f"Candidatos a titulares identificados: {len(starters_candidates)}")
        print(f"\nTop candidatos:")
        print(starters_candidates.head(11))
        
        return starters_candidates
    
    def visualize_sample_heatmap(self, player_name=None):
        """Crea mapa de calor de muestra para validar coordenadas."""
        merged = self.test_player_events_merge()
        if merged is None:
            return
            
        # Seleccionar jugador
        if player_name is None:
            top_player = merged.groupby('player_name').size().idxmax()
            player_name = top_player
        
        player_data = merged[merged['player_name'] == player_name]
        
        if len(player_data) == 0:
            print(f"No se encontraron datos para {player_name}")
            return
            
        print(f"\nGENERANDO MAPA DE CALOR DE PRUEBA: {player_name}")
        print("=" * 50)
        
        # Crear pitch
        pitch = Pitch(pitch_type='opta', pitch_color='#22312b', line_color='#c7d5cc')
        fig, ax = pitch.draw(figsize=(12, 8))
        
        # Mapa de calor
        if len(player_data) >= 5:
            pitch.kdeplot(player_data['x'], player_data['y'], ax=ax, 
                         cmap='Reds', fill=True, levels=15, alpha=0.7)
        else:
            pitch.scatter(player_data['x'], player_data['y'], ax=ax, 
                         s=100, c='red', alpha=0.8)
        
        ax.set_title(f'{player_name}\nMapa de Calor de Prueba ({len(player_data)} eventos)',
                    fontsize=14, color='white')
        fig.set_facecolor('#22312b')
        
        plt.show()
        
        # Estadísticas del jugador
        print(f"Eventos del jugador: {len(player_data)}")
        print(f"Posición: {player_data['position'].iloc[0] if 'position' in player_data.columns else 'N/A'}")
        print(f"Rango temporal: {player_data['minute'].min()}-{player_data['minute'].max()} min")
        print(f"Coordenadas X: {player_data['x'].min():.1f} - {player_data['x'].max():.1f}")
        print(f"Coordenadas Y: {player_data['y'].min():.1f} - {player_data['y'].max():.1f}")
        
        return player_data
    
    def run_complete_debug(self):
        """Ejecuta debug completo de la estructura de datos."""
        print("INICIANDO DEBUG COMPLETO")
        print("=" * 60)
        
        # Cargar todos los CSVs
        self.load_all_csvs()
        
        # Analizar cada componente
        self.analyze_match_meta()
        self.analyze_events_structure()
        self.analyze_players_structure()
        self.analyze_formations_timeline()
        self.analyze_positions_timeline()
        
        # Pruebas de integración
        self.test_player_events_merge()
        self.identify_starters_by_events()
        
        # Visualización de prueba
        self.visualize_sample_heatmap()
        
        print(f"\n" + "=" * 60)
        print("DEBUG COMPLETO FINALIZADO")
        print("=" * 60)
        
        return self.data_loaded

# =====================================================================
# FUNCIONES DE USO DIRECTO
# =====================================================================

def debug_match_structure(match_path):
    """Función directa para debuggear un partido específico."""
    debugger = DataStructureDebugger(match_path)
    return debugger.run_complete_debug()

def quick_debug(match_path, player_name=None):
    """Debug rápido con visualización."""
    debugger = DataStructureDebugger(match_path)
    debugger.load_all_csvs()
    debugger.test_player_events_merge()
    debugger.visualize_sample_heatmap(player_name)
    
    return debugger

# =====================================================================
# EJECUCIÓN DE EJEMPLO
# =====================================================================

# Ruta del partido de ejemplo que proporcionaste
MATCH_PATH = r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\matchcenter\MatchCenter\Competition\Season\20250815_Girona_vs_Rayo_Vallecano_1913916"

print("DEBUGGER DE ESTRUCTURA DE DATOS")
print("=" * 40)
print("Uso:")
print("• debug_match_structure(MATCH_PATH): Debug completo")
print("• quick_debug(MATCH_PATH, 'Nombre_Jugador'): Debug rápido")
print(f"\nRuta configurada: {MATCH_PATH}")

In [ ]:
debug_match_structure(MATCH_PATH)

In [ ]:
import pandas as pd
from pathlib import Path

def debug_player_detection(base_path, target_player="Pablo Fornals"):
    """Debug específico para entender por qué no se detecta un jugador."""
    
    print(f"DEBUGGEANDO DETECCIÓN DE: {target_player}")
    print("=" * 60)
    
    base_path = Path(base_path)
    found_player = False
    player_info = []
    
    # Buscar en todos los partidos
    match_dirs = []
    for item in base_path.rglob("*"):
        if item.is_dir() and (item / 'csv' / 'events.csv').exists():
            match_dirs.append(item)
    
    print(f"Partidos encontrados: {len(match_dirs)}")
    
    for i, match_dir in enumerate(match_dirs):
        print(f"\n[{i+1}/{len(match_dirs)}] Analizando: {match_dir.name}")
        
        # Cargar datos
        csv_dir = match_dir / 'csv'
        
        try:
            events_df = pd.read_csv(csv_dir / 'events.csv')
            players_df = pd.read_csv(csv_dir / 'players.csv')
            
            # Buscar el jugador específico
            player_match = players_df[
                players_df['player_name'].str.contains(target_player, case=False, na=False)
            ]
            
            if len(player_match) > 0:
                found_player = True
                player_row = player_match.iloc[0]
                
                print(f"  ✓ ENCONTRADO: {player_row['player_name']}")
                print(f"    Titular: {player_row.get('isFirstEleven', 'N/A')}")
                print(f"    Posición: {player_row.get('position', 'N/A')}")
                print(f"    Equipo: {player_row.get('team_name', 'N/A')}")
                print(f"    Rating: {player_row.get('rating', 'N/A')}")
                
                # Contar eventos
                player_events = events_df[events_df['playerId'] == player_row['player_id']]
                events_count = len(player_events)
                print(f"    Eventos: {events_count}")
                
                # Verificar por qué no se incluye
                is_starter = player_row.get('isFirstEleven', False)
                has_enough_events = events_count >= 5
                
                print(f"    Pasa filtro titular: {is_starter}")
                print(f"    Pasa filtro eventos (>=5): {has_enough_events}")
                
                if is_starter and has_enough_events:
                    print(f"    ✓ DEBERÍA ser incluido en el análisis")
                else:
                    print(f"    ✗ NO pasa filtros del sistema")
                
                player_info.append({
                    'match': match_dir.name,
                    'name': player_row['player_name'],
                    'is_starter': is_starter,
                    'events': events_count,
                    'position': player_row.get('position', 'N/A'),
                    'team': player_row.get('team_name', 'N/A'),
                    'rating': player_row.get('rating', 0)
                })
            else:
                print(f"  - No encontrado en este partido")
                
        except Exception as e:
            print(f"  ✗ Error procesando partido: {e}")
            continue
    
    if not found_player:
        print(f"\n❌ {target_player} NO encontrado en ningún partido")
        print("\nJugadores disponibles en los primeros 3 partidos:")
        
        # Mostrar jugadores disponibles
        for i, match_dir in enumerate(match_dirs[:3]):
            try:
                players_df = pd.read_csv(match_dir / 'csv' / 'players.csv')
                starters = players_df[players_df['isFirstEleven'] == True]
                print(f"\nPartido {i+1}: {match_dir.name}")
                for _, player in starters.head(10).iterrows():
                    print(f"  {player['player_name']} ({player.get('position', 'N/A')})")
            except:
                continue
    else:
        print(f"\n✅ {target_player} encontrado en {len(player_info)} partido(s)")
        
        # Resumen
        df_info = pd.DataFrame(player_info)
        print(f"\nResumen:")
        print(f"  Partidos como titular: {sum(df_info['is_starter'])}")
        print(f"  Eventos promedio: {df_info['events'].mean():.1f}")
        print(f"  Rating promedio: {df_info[df_info['rating'] > 0]['rating'].mean():.2f}")
    
    return player_info

def show_all_available_players(base_path, max_matches=6):
    """Muestra todos los jugadores disponibles con sus estadísticas."""
    
    print("JUGADORES DISPONIBLES EN EL SISTEMA")
    print("=" * 50)
    
    base_path = Path(base_path)
    player_stats = {}
    
    # Buscar partidos
    match_dirs = []
    for item in base_path.rglob("*"):
        if item.is_dir() and (item / 'csv' / 'events.csv').exists():
            match_dirs.append(item)
    
    match_dirs = match_dirs[:max_matches]
    
    for match_dir in match_dirs:
        try:
            csv_dir = match_dir / 'csv'
            events_df = pd.read_csv(csv_dir / 'events.csv')
            players_df = pd.read_csv(csv_dir / 'players.csv')
            
            # Merge para obtener eventos por jugador
            events_with_players = events_df.merge(
                players_df[['player_id', 'player_name', 'position', 'isFirstEleven', 'team_name']],
                left_on='playerId',
                right_on='player_id',
                how='left'
            )
            
            # Procesar cada jugador titular
            starters = players_df[players_df['isFirstEleven'] == True]
            for _, player in starters.iterrows():
                name = player['player_name']
                player_events = events_with_players[
                    events_with_players['player_name'] == name
                ]
                events_count = len(player_events)
                
                if events_count >= 5:  # Aplicar mismo filtro que el sistema
                    if name not in player_stats:
                        player_stats[name] = {
                            'matches': 0,
                            'total_events': 0,
                            'position': player.get('position', 'N/A'),
                            'team': player.get('team_name', 'N/A')
                        }
                    
                    player_stats[name]['matches'] += 1
                    player_stats[name]['total_events'] += events_count
                    
        except Exception as e:
            print(f"Error procesando {match_dir.name}: {e}")
            continue
    
    # Mostrar resultados
    sorted_players = sorted(player_stats.items(), 
                           key=lambda x: (x[1]['matches'], x[1]['total_events']), 
                           reverse=True)
    
    print(f"Jugadores encontrados: {len(sorted_players)}")
    print(f"Partidos analizados: {len(match_dirs)}")
    print("\nTop 20 jugadores por actividad:")
    print("-" * 70)
    
    for i, (name, stats) in enumerate(sorted_players[:20], 1):
        avg_events = stats['total_events'] / stats['matches']
        print(f"{i:2d}. {name:<25} | {stats['matches']} partidos | "
              f"{avg_events:5.1f} eventos/partido | {stats['position']}")
    
    return sorted_players

# Ejecutar debug
BASE_PATH = r"C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\matchcenter\MatchCenter\Competition\Season"

print("Ejecutando debug...")
player_info = debug_player_detection(BASE_PATH, "Pablo Fornals")

print("\n" + "="*60)
print("Mostrando todos los jugadores disponibles...")
all_players = show_all_available_players(BASE_PATH, 6)